# Netra-One — Trajectory Forecasting & Predictive Breach Alerts (Colab)

Runs the full pipeline end-to-end on a free Colab runtime (T4 GPU or CPU):
synthetic tracks → LSTM seq2seq training → ADE/FDE benchmark vs Kalman/polynomial → predictive perimeter-breach alerts.

*Runtime → Change runtime type → T4 GPU* is optional; everything also runs on CPU.

In [ ]:
REPO_URL = "https://github.com/kvlokesh2005-arch/bogathi-netraone-trajectory-prediction.git"
!git clone -q $REPO_URL netraone || echo "already cloned"
%cd netraone
!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Generate the synthetic intruder-track dataset

In [ ]:
!python generate_synthetic_tracks.py --num-tracks 3000

## 2. Train the LSTM encoder-decoder (~2-3 min on T4, ~10 min on CPU)
Skip this cell to use the committed checkpoint in `checkpoints/`.

In [ ]:
!python train.py --cell lstm --epochs 50

## 3. ADE / FDE benchmark + alert precision / recall on the test split

In [ ]:
!python evaluate.py

## 4. Predictive breach alerts + trajectory plots

In [ ]:
!python predict_breach.py --model lstm --num-samples 9
!python predict_breach.py --model kalman --track-id 104
!python predict_breach.py --model lstm --demo --delay 0

## 5. Unit tests

In [ ]:
!pytest -q

## 6. Figures

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("sample_outputs/results_table.md").read()))
display(Markdown(open("sample_outputs/alert_metrics.md").read()))
for f in ["trajectory_forecast.png", "comparison_grid.png", "forecast_grid_lstm.png",
          "loss_curves.png", "ade_fde_bar.png", "error_vs_step.png"]:
    display(Image(f"sample_outputs/{f}"))